In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/home/xiaowenz/finetune')

In [ ]:
import datasets
import random

random.seed(903)

def filter_OE(item):
  if 'answer' in item:
    return item['answer'].lower() in ['yes', 'no']
  return (
    item['answer_type'] == 'yes/no'
    and len(item['label']['ids']) == 1
  )

def extract_label(item):
  item['label'] = item['label']['ids'][0]
  return item
ds = datasets.load_dataset('Graphcore/vqa').filter(filter_OE).map(extract_label)
to_drop = []
for k, v in ds.items():
  try:
    ds[k] = (
      v
      .select(random.sample(range(len(v)), 10000))
      .cast_column('image_id', datasets.features.Image())
      .rename_column('image_id', 'image')
    )
  except Exception as e:
    print(k)
    to_drop.append(k)

for k in to_drop:
  ds.pop(k)

ds.push_to_hub('withcomment/vqa_10k')

In [ ]:
def filter_OE(item):
  if 'answer' in item:
    return item['answer'].lower() in ['yes', 'no']
  return (
    item['answer_type'] == 'yes/no'
    and len(item['label']['ids']) == 1
  )

In [ ]:
import torch
from tqdm import tqdm
import transformers
from torch import nn
import json

transformers.enable_full_determinism(903)
processor = transformers.AutoProcessor.from_pretrained('/scratch/xiaowenz/checkpoints/Qwen2_5_3', use_fast=True)
tokenizer = processor.tokenizer


def eval_model(model, ds, collate_fn, tokenizer, sample_idx=range(1000)) -> dict[str, float]:
  correct = 0
  total = 0
  invalid = 0
  confidence = 0
  with torch.inference_mode():
    for item in tqdm(ds.ds.select(sample_idx)):
      if item['answer'] not in ['yes', 'no']:
        continue
      inputs = collate_fn([item])
      outputs = model(**inputs.to(model.device))
      probs = torch.softmax(outputs.logits[0, -1], dim=-1)
      logit, output_id = probs.max(dim=-1)
      outputs = tokenizer.decode(output_id, skip_special_tokens=True)
      output = outputs.split("assistant")[-1].strip().strip("\n")
      answer = item['answer']
      total += 1
      if output.lower() not in ['yes', 'no']:
        invalid += 1
        continue
      if output.lower() == answer.lower():
        correct += 1
        confidence += logit.exp().item()
  accuracy = correct / total if total > 0 else 0
  confidence /= total if total > 0 else 1
  valid = total - invalid
  return {
    'total': total,
    'invalid': invalid,
    'acc': accuracy,
    'acc_wo_invalid': correct / valid if valid > 0 else 0,
    'confidence': confidence
  }

In [ ]:
from copy import copy
from qwenvl.argument import DataArguments, ProcessingArguments
from qwenvl.data.conversation import AllPromptAdder, FirstPromptAdder
from qwenvl.data.preprocess import FilterStrategy
from qwenvl.module import create_strategies, create_module
from qwenvl.train import set_processor


proc_args_default = ProcessingArguments(
    use_chat_template=True,
    sys_prompt="default",
    usr_prompt='',
    add_generation_prompt=True,
    use_bf16=True,
    resize_img=False,
    base_interval=0.5,
    for_training=False
)
yes_no_modifier = AllPromptAdder('Choose from "yes" or "no".\n', -1, 'user')

In [ ]:

data_args_bad = DataArguments(
    dataset_use='vqa',
    split='train',
    packing=False,
    model_max_length=3072,)
processor = set_processor(
    processor, proc_args_default, data_args_bad)

strategies_bad, cp_bad, ip_bad = create_strategies(
    processor, data_args_bad, proc_args_default, rank=0, additional_kwargs={'shuffle': 1})
cp_bad.modifiers.append(yes_no_modifier)
ds_bad, collate_bad = create_module(
    data_args_bad, strategies_bad, cp_bad, ip_bad)

data_args_good = DataArguments(
    dataset_use='path_vqa',
    split='validation',
    packing=False,
    model_max_length=3072,)
strategies_good, cp_good, ip_good = create_strategies(
    processor, data_args_good, proc_args_default, rank=0, additional_kwargs={'shuffle': 0})
strategies_good.append(FilterStrategy(filter_OE))
cp_good.modifiers.append(yes_no_modifier)
ds_good, collate_good = create_module(
    data_args_good, strategies_good, cp_good, ip_good)

data_args_test = DataArguments(
    dataset_use='path_vqa',
    split='test',
    packing=False,
    portion=1000,
    model_max_length=3072,)
strategies_test, cp_test, ip_test = create_strategies(
    processor, data_args_test, proc_args_default, rank=0, additional_kwargs={'shuffle': 0})
strategies_test.append(FilterStrategy(filter_OE))
cp_test.modifiers.append(yes_no_modifier)
ds_test, collate_test = create_module(
    data_args_test, strategies_test, cp_test, ip_test)

In [ ]:
import copy
data_args_test = DataArguments(
    dataset_use='path_vqa',
    split='test',
    packing=False,
    portion=1000,
    model_max_length=3072,)

proc_args_v3_5 = copy.deepcopy(proc_args_default)
proc_args_v3_5.sys_prompt = "v3_5"

strategies_v3_5, cp_v3_5, ip_v3_5 = create_strategies(
    processor, data_args_test, proc_args_v3_5, rank=0, additional_kwargs={'shuffle': 0})
strategies_v3_5.append(FilterStrategy(filter_OE))
cp_v3_5.modifiers.append(yes_no_modifier)
ds_v3_5, collate_v3_5 = create_module(
    data_args_test, strategies_v3_5, cp_v3_5, ip_v3_5)

In [ ]:
prompt = 'default'
model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
    f'/scratch/xiaowenz/checkpoints/Qwen2_5_3_path_vqa_{prompt}/last', torch_dtype=torch.bfloat16, attn_implementation='flash_attention_2').to('cuda')
v3_5 = eval_model(model, ds_test, collate_test, tokenizer)
with open(f'experiments/path_vqa_{prompt}_last.json', 'w') as f:
  json.dump(v3_5, f)
del model
torch.cuda.empty_cache()

for prompt in ['v3_5']:
  for checkpoint_number in [62, 124, 186, 248, 308]:
    model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
        f'/scratch/xiaowenz/checkpoints/Qwen2_5_3_path_vqa_{prompt}/checkpoint-{checkpoint_number}', torch_dtype=torch.bfloat16, attn_implementation='flash_attention_2').to('cuda')
    v3_5 = eval_model(model, ds_v3_5, collate_v3_5, tokenizer)
    with open(f'experiments/path_vqa_{prompt}_{checkpoint_number}.json', 'w') as f:
      json.dump(v3_5, f)
  del model
  torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 10%|█         | 104/1000 [00:21<03:09,  4.73it/s]

In [ ]:
from typing import Callable
from tqdm import tqdm


def add_hooks(model, param_dict: dict[str, torch.Tensor]) -> Callable:
  def getActivation(name: str):
    def hook(module, input, output):
      activations = input[0]
      weights = module.weight.data
      # Compute the norm of activations along dim=1
      activations_norm = activations.norm(p=2, dim=1).to(torch.bfloat16)
      # Multiply activations by the absolute value of weights
      modified_output = activations_norm * torch.abs(weights)
      param_dict[name] += modified_output.detach()

    return hook

  handles = []
  for name, module in model.named_modules():
    if (
        name.startswith('model.language_model')
        and isinstance(module, (nn.Linear))
        and 'mlp' in name
    ):
      param_dict[name] = torch.zeros_like(module.weight.data).to(model.device)
      hook_fn = getActivation(name)
      handles.append(module.register_forward_hook(hook_fn))

  def remove_hooks():
    for handle in handles:
      handle.remove()
  return remove_hooks


In [ ]:
import random


def get_important(model, ds, collate_fn, ratio, n_samples=1000):
  param_dict = {}  
  remove_hooks = add_hooks(model, param_dict)
  with torch.inference_mode():
    for item in tqdm(ds.ds.select(random.sample(range(len(ds.ds)), n_samples))):
      model(**collate_fn([item]).to(model.device))

  mask_dict = {}

  for k, v in param_dict.items():
    # don't count classifier layer
    if "embed" in k:
      mask_dict[k] = torch.ones_like(v, dtype=torch.int8).to(v.device)
      continue
    topk = int(v.numel() * ratio)
    tensor = v.view(-1)
    top_pos = torch.topk(torch.abs(tensor), topk)[1]
    mask_dict[k] = torch.zeros_like(tensor, device=tensor.device)
    mask_dict[k][top_pos] = 1
    mask_dict[k] = mask_dict[k].reshape(v.shape).to(tensor.device)

  remove_hooks()
  del param_dict
  torch.cuda.empty_cache()
  
  return mask_dict

In [ ]:
def get_specific_important(good_params, bad_params):
  specific_important = {}
  for k, v in good_params.items():
    # Specifically important if important (1) in good, unimportant (0) in bad.
    specific_important[k] = (
      (good_params[k] - bad_params[k]) > 0).to(torch.bool)
  del good_params
  del bad_params
  torch.cuda.empty_cache()
  return specific_important


In [ ]:
import random



def scale_important(model, important_mask, factor):
  for k, v in important_mask.items():
    model.state_dict()[f'{k}.weight'][v] *= factor
  return model


In [ ]:
good_percents: dict[float, dict] = {
  0: {'performance': {}},
  0.01: {'performance': {}},
  0.015: {'performance': {}},
  0.02: {'performance': {}},
  0.05: {'performance': {}}
}
n_samples = 1000
sample_idx = random.sample(range(len(ds_test)), n_samples)

for portion in good_percents:
  model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
    '/scratch/xiaowenz/checkpoints/Qwen2_5_3', attn_implementation='flash_attention_2', device_map='cuda:0', torch_dtype=torch.bfloat16)
  
  important_good = get_important(model, ds_good, collate_good, portion, n_samples)
  important_bad = get_important(model, ds_bad, collate_bad, portion, n_samples)
  mask = get_specific_important(important_good, important_bad)
  total, will_scale = 0, 0
  for k, v in mask.items():
    total += v.numel()
    will_scale += v.sum().item()
  for factor in [1.01, 1.05, 1.1, 1.15]:
    del model
    torch.cuda.empty_cache()
    model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
      '/scratch/xiaowenz/checkpoints/Qwen2_5_3', attn_implementation='flash_attention_2', device_map='cuda:0', torch_dtype=torch.bfloat16)
    model = scale_important(model, mask, factor)
    good_percents[portion]['performance'][factor] = eval_model(model, ds_test, collate_test, tokenizer, sample_idx)
  good_percents[portion]['total_params'] = total
  good_percents[portion]['scaled_params'] = will_scale
  good_percents[portion]['scaled_portion'] = will_scale / total
  good_percents[portion]['rel_scaled_portion'] = will_scale / (total * portion) if portion > 0 else 1

  del model
  del important_good
  del important_bad
  del mask
  torch.cuda.empty_cache()

good_percents

In [ ]:
import numpy as np


best_portion = 0
best_mean_acc_wo_invalid = 0
for portion, stats in good_percents.items():
  mean_acc_wo_invalid = np.mean([v['acc_wo_invalid'] for v in stats['performance'].values()])
  if mean_acc_wo_invalid > best_mean_acc_wo_invalid:
    best_mean_acc_wo_invalid = mean_acc_wo_invalid
    best_portion = portion

model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
  '/scratch/xiaowenz/checkpoints/Qwen2_5_3', attn_implementation='flash_attention_2', device_map='cuda:0', torch_dtype=torch.bfloat16)

important_good = get_important(model, ds_good, collate_good, best_portion, n_samples)
important_bad = get_important(model, ds_bad, collate_bad, best_portion, n_samples)
mask = get_specific_important(important_good, important_bad)
torch.save(mask, f'important_mask_path_vqa_{str(best_portion).replace(".", "_")}.pt')

In [ ]:
import torch
import transformers

model = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
  "/scratch/xiaowenz/checkpoints/Qwen2_5_3", attn_implementation='flash_attention_2', torch_dtype=torch.bfloat16)
model_default = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
  "/scratch/xiaowenz/checkpoints/Qwen2_5_3_path_vqa_default/last", attn_implementation='flash_attention_2', torch_dtype=torch.bfloat16)
model_custom = transformers.Qwen2_5_VLForConditionalGeneration.from_pretrained(
  "/scratch/xiaowenz/checkpoints/Qwen2_5_3_path_vqa_v2_0/last", attn_implementation='flash_attention_2', torch_dtype=torch.bfloat16)
mask = torch.load(f'important_mask_path_vqa_0_01.pt')

In [ ]:
for k, v in mask.items():
  print(k, v.to(float).mean())

In [ ]:
total = 0
r_default = 0
r_custom = 0

def get_r_for_param(param_base, param_finetuned, mask):
  if torch.sum(mask) == 0 or torch.sum(~mask) == 0:
    return torch.inf
  
  diff = (param_finetuned - param_base) ** 2
  in_group_avg_squared_diff = diff[mask].mean()
  out_group_avg_squared_diff = diff.mean()
  return in_group_avg_squared_diff / out_group_avg_squared_diff


for k, v in mask.items():
  if 'gate' in k:
    continue
  v = v.cpu()
  total += v.numel()
  param_base = model.state_dict()[f'{k}.weight'].cpu().to(torch.float32)
  param_default = model_default.state_dict()[f'{k}.weight'].cpu().to(torch.float32)
  param_custom = model_custom.state_dict()[f'{k}.weight'].cpu().to(torch.float32)
  r_default += get_r_for_param(param_base, param_default, v) * v.numel()
  r_custom += get_r_for_param(param_base, param_custom, v) * v.numel()

r_default / total, r_custom / total